# 02 - Synthetic gap layer (SPEC section 9.1)
Generates, with `dbldatagen`, only the data that public sources cannot provide. Everything is generated **over real companies** from the ASIC Company Dataset (P1) and is conditioned on whether each company actually collapsed, so the signal is realistic.

| Dataset | Gap it fills |
|---|---|
| `curated_slice` | The ~5k focus entities (real P1 companies: licensees, collapsed, healthy) |
| `persons`, `directorships` | Director appointments (paid ASIC extracts). A serial-director pool creates phoenix networks; some names come from the real banned list (P4) |
| `ato_tax_profile` | ATO debts, lodgement compliance, STP payroll |
| `austrac_smr` | AUSTRAC suspicious matter reports (restricted; masked downstream) |
| `apra_prudential_flags` | APRA prudential flags for licensees |
| `afca_complaints`, `enforcement_alternatives`, `published_notices` | Web-only sources P15, P8, P7 |
| `breach_reports` | Intake queue items |
| `historical_matters` | Internal litigation cost, duration and recovery history |
| `cases`, `case_parties`, `teams`, `users` | Open case files, including 3 story cases |

Every row carries `provenance = 'synthetic'`. Output: `/Volumes/<catalog>/<schema>/<raw_volume>/synthetic/<dataset>/` (parquet, regenerated deterministically from `random_seed`).

In [0]:
%pip install -q dbldatagen
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("raw_volume", "raw")
dbutils.widgets.text("curated_slice_size", "5000")
dbutils.widgets.text("num_cases", "40")
dbutils.widgets.text("random_seed", "42")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
RAW_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{dbutils.widgets.get('raw_volume')}"
N = int(dbutils.widgets.get("curated_slice_size"))
NUM_CASES = int(dbutils.widgets.get("num_cases"))
SEED = int(dbutils.widgets.get("random_seed"))
assert CATALOG and SCHEMA and NUM_CASES >= 3

In [0]:
import os, re
import dbldatagen as dg
from pyspark.sql import functions as F, Window as W

def latest(source_id):
    base = f"{RAW_ROOT}/public/{source_id}"
    snaps = sorted(d.name for d in os.scandir(base) if d.name.startswith("snapshot="))
    assert snaps, f"no snapshot for {source_id}; run 01_ingest_public_data first"
    return f"{base}/{snaps[-1]}"

def norm_cols(df):
    """Strip BOM, lower-case, snake_case."""
    return df.toDF(*[re.sub(r"[^0-9a-z]+", "_", c.replace("\ufeff", "").strip().lower()).strip("_") for c in df.columns])

def read_tsv(path):
    return norm_cols(spark.read.option("header", True).option("sep", "\t").option("quote", "\u0000").csv(path))

def gen(name, rows, cols, partitions=8):
    """Build a dbldatagen spec from (name, type, kwargs) tuples. Keeps the generated 'id'."""
    s = dg.DataGenerator(spark, name=name, rows=rows, partitions=partitions,
                         randomSeedMethod="hash_fieldname", randomSeed=SEED).withIdOutput()
    for c, t, kw in cols:
        s = s.withColumn(c, t, **kw)
    return s.build()

# Continuous uniforms need an explicit step in dbldatagen (otherwise only 0.0/1.0 are produced).
# Avoid apostrophes in value lists (they break dbldatagen's generated SQL).
U = lambda n: (n, "double", dict(minValue=0.0, maxValue=1.0, step=0.0001, random=True))

def write(df, dataset, provenance="synthetic"):
    path = f"{RAW_ROOT}/synthetic/{dataset}"
    out = (df.withColumn("provenance", F.lit(provenance))
             .withColumn("generated_at", F.current_timestamp())
             .withColumn("random_seed", F.lit(SEED)))
    out.write.mode("overwrite").parquet(path)
    n = spark.read.parquet(path).count()
    print(f"{dataset:28s} {n:>10,} rows")
    return n

FIRST = ["James","Olivia","William","Charlotte","Jack","Amelia","Noah","Isla","Thomas","Mia","Oliver","Grace","Lucas","Chloe","Henry","Zoe","Ethan","Ruby","Daniel","Sophie","Wei","Mei","Arjun","Priya","Mohammed","Fatima","Minh","Linh","Luca","Sofia"]
LAST = ["Smith","Jones","Williams","Brown","Wilson","Taylor","Nguyen","Johnson","Martin","White","Anderson","Thompson","Walker","Harris","Lee","Ryan","Robinson","Kelly","King","Chen","Wang","Singh","Patel","Tran","Kaur","Rossi","Papadopoulos","Murphy","Campbell","Clarke"]
STATES, STATE_W = ["NSW","VIC","QLD","WA","SA","TAS","ACT","NT"], [32,26,20,10,7,2,2,1]

In [0]:
companies = (read_tsv(latest("p01_asic_companies") + "/*.csv")
             .filter(F.col("current_name_indicator") == "Y")
             .select("acn", "abn", "company_name", "type", "status",
                     F.to_date("date_of_registration", "dd/MM/yyyy").alias("reg_date"),
                     F.to_date("date_of_deregistration", "dd/MM/yyyy").alias("dereg_date"))
             .dropDuplicates(["acn"]))

# Licensees (AFS + credit) matched on ACN (= last 9 digits of a company ABN)
lic_ids = (read_tsv(latest("p03_asic_afs_licensees") + "/*.tsv").select(F.col("afs_lic_abn_acn").alias("id"))
           .unionByName(read_tsv(latest("p03_asic_credit_licensees") + "/*.tsv").select(F.col("cred_lic_abn_acn").alias("id")))
           .select(F.regexp_replace("id", "[^0-9]", "").alias("id")).filter(F.length("id") >= 9)
           .select(F.substring(F.lpad("id", 11, "0"), -9, 9).alias("acn")).distinct())

ten_years_ago = F.add_months(F.current_date(), -120)
cand = (companies.filter(F.col("type").isin("APTY", "APUB") & (F.col("reg_date") >= "2000-01-01"))
        .join(lic_ids.withColumn("is_licensee", F.lit(1)), "acn", "left").fillna({"is_licensee": 0})
        .withColumn("collapsed", ((F.col("status") == "EXAD") |
                                  ((F.col("status") == "DRGD") & (F.col("dereg_date") >= ten_years_ago))).cast("int"))
        .filter((F.col("status") == "REGD") | (F.col("collapsed") == 1))
        .withColumn("h", F.xxhash64("acn", F.lit(SEED))))

# Quotas: 20% licensees, 35% collapsed, remainder healthy
q_lic, q_col = int(N * 0.20), int(N * 0.35)
lic = cand.filter("is_licensee = 1").orderBy("h").limit(q_lic).withColumn("slice_reason", F.lit("licensee"))
col = cand.filter("is_licensee = 0 AND collapsed = 1").orderBy("h").limit(q_col).withColumn("slice_reason", F.lit("collapsed"))
healthy = cand.filter("is_licensee = 0 AND collapsed = 0").orderBy("h").limit(N - q_lic - q_col).withColumn("slice_reason", F.lit("healthy"))

slc = (lic.unionByName(col).unionByName(healthy)
       .withColumn("idx", F.row_number().over(W.orderBy("h")) - 1)
       # EXAD has no public date in P1: use a synthetic appointment date (P7 notices are web-only)
       .withColumn("collapse_date", F.when(F.col("status") == "DRGD", F.col("dereg_date"))
                                    .when(F.col("status") == "EXAD", F.date_sub(F.current_date(), (F.abs(F.col("h")) % 1460).cast("int"))))
       # Realism: agency distress signals are noisy. ~70% of collapsing companies show them, ~10% of healthy ones do too.
       .withColumn("_pct", F.abs(F.xxhash64("acn", F.lit(SEED + 7))) % 100)
       .withColumn("distressed", F.when(F.col("collapsed") == 1, F.col("_pct") < 70).otherwise(F.col("_pct") < 10).cast("int"))
       .drop("h", "_pct").cache())
N = slc.count()
print("slice size", N)
display(slc.groupBy("slice_reason", "status").count())
write(slc.select("acn", "abn", "company_name", "status", "slice_reason", "is_licensee"), "curated_slice", provenance="public")

In [0]:
NP = int(N * 1.3)               # people
POOL = max(int(NP * 0.05), 10)  # serial directors (sit on many collapsed companies)

persons = gen("persons", NP, [
    ("given_name", "string", dict(values=FIRST, random=True)),
    ("middle_initial", "string", dict(values=list("ABCDEFGHJKLMNPRSTW"), random=True)),
    ("family_name", "string", dict(values=LAST, random=True)),
    ("dob", "date", dict(data_range=dg.DateRange("1945-01-01 00:00:00", "2000-12-31 00:00:00", "days=1"), random=True)),
    ("state", "string", dict(values=STATES, weights=STATE_W, random=True)),
    ("postcode", "int", dict(minValue=2000, maxValue=7999, random=True)),
    U("u_ban")]).withColumnRenamed("id", "person_idx")

# Real banned/disqualified names (P4) are re-used for ~30% of the serial-director pool.
# A name match on the banned list does NOT prove identity - entity resolution must review these.
banned = (read_tsv(latest("p04_asic_banned_persons") + "/*.tsv")
          .select(F.split("bd_per_name", ",\\s*").alias("p")).filter(F.size("p") == 2)
          .select(F.initcap(F.col("p")[0]).alias("b_family"), F.initcap(F.split(F.col("p")[1], " ")[0]).alias("b_given"))
          .withColumn("b_idx", F.row_number().over(W.orderBy(F.xxhash64("b_family", "b_given", F.lit(SEED)))) - 1))
persons = (persons.join(banned, (persons.person_idx == banned.b_idx) & (persons.person_idx < POOL) & (persons.u_ban < 0.3), "left")
           .withColumn("given_name", F.coalesce("b_given", "given_name"))
           .withColumn("family_name", F.coalesce("b_family", "family_name"))
           .withColumn("name_from_banned_register", F.col("b_family").isNotNull())
           .withColumn("person_id", F.format_string("SP%07d", "person_idx"))
           .drop("b_given", "b_family", "b_idx", "u_ban"))

d = gen("directorships", N * 2, [
    ("company_idx", "long", dict(minValue=0, maxValue=N - 1, random=True)),
    ("role", "string", dict(values=["Director", "Secretary", "Director and Secretary"], weights=[70, 10, 20], random=True)),
    ("tenure_days", "int", dict(minValue=90, maxValue=4000, random=True)),
    U("u1"), U("u2")])
directorships = (d.join(slc.select("idx", "acn", "collapsed", "reg_date", "collapse_date"), d.company_idx == slc.idx)
    .withColumn("person_idx", F.when((F.col("collapsed") == 1) & (F.col("u1") < 0.5), (F.col("u2") * POOL).cast("long"))
                               .otherwise((POOL + F.col("u2") * (NP - POOL - 1)).cast("long")))
    .withColumn("appointed_date", F.greatest("reg_date", F.date_sub(F.coalesce("collapse_date", F.current_date()), "tenure_days")))
    .withColumn("ceased_date", F.col("collapse_date"))
    .withColumn("person_id", F.format_string("SP%07d", "person_idx"))
    .select(F.format_string("SD%08d", "id").alias("directorship_id"), "person_id", "acn", "role", "appointed_date", "ceased_date")
    .dropDuplicates(["person_id", "acn"]))

write(persons, "persons")
write(directorships, "directorships")

In [0]:
Q = 16  # quarters of history per company (4 years), ending at the collapse date (or today)
DISTRESS_Q = 8  # collapsing companies deteriorate over their final 8 quarters
# 16 quarters lets 24-month collapse labels be observed for healthy companies too (as_of + 24m <= today)
ato = gen("ato", N * Q, [
    ("company_idx", "long", dict(expr=f"floor(id / {Q})")),
    ("q", "int", dict(expr=f"cast(id % {Q} as int)")),
    U("u1"), U("u2"), U("u3"), U("u4")])
ato = (ato.join(slc.select("idx", "acn", "distressed", "collapse_date"), ato.company_idx == slc.idx)
    .withColumn("period_end", F.last_day(F.add_months(F.coalesce("collapse_date", F.current_date()), (F.col("q") - Q) * 3)))
    .withColumn("base", F.exp(F.lit(8.5) + F.col("u1") * 4.5))
    .withColumn("k", F.greatest(F.col("q") - (Q - DISTRESS_Q), F.lit(0)))  # quarters into the distress window
    # Collapsing companies: debt compounds ~35%/quarter; lodgement and payroll deteriorate
    .withColumn("ato_debt_aud", F.round(F.when(F.col("distressed") == 1, F.col("base") * F.pow(F.lit(1.35), "k"))
                                        .otherwise(F.col("base") * (0.9 + F.col("u2") * 0.2) * F.when(F.col("u4") < 0.6, 0.0).otherwise(1.0)), 2))
    .withColumn("lodgement_on_time_pct", F.round(F.when(F.col("distressed") == 1, 0.85 - F.col("k") * 0.05 - F.col("u3") * 0.2)
                                                 .otherwise(0.75 + F.col("u3") * 0.25), 3))
    .withColumn("stp_headcount", F.greatest(F.lit(0), F.round(F.exp(1 + F.col("u2") * 4) * F.when(F.col("distressed") == 1, 1 - F.col("k") * 0.08).otherwise(1.0)).cast("int")))
    .withColumn("gst_registered", ~((F.col("distressed") == 1) & (F.col("q") == Q - 1) & (F.col("u4") < 0.5)))
    .select("acn", "period_end", "ato_debt_aud", "lodgement_on_time_pct", "stp_headcount", "gst_registered"))
write(ato, "ato_tax_profile")

smr = gen("austrac", int(N * 0.6), [
    ("company_idx", "long", dict(minValue=0, maxValue=N - 1, random=True)),
    ("report_date", "date", dict(data_range=dg.DateRange("2022-01-01 00:00:00", "2026-10-31 00:00:00", "days=1"), random=True)),
    ("smr_type", "string", dict(values=["Structuring", "Unusual cash", "Rapid movement offshore", "Third party deposits", "Identity concerns"], weights=[25, 25, 20, 20, 10], random=True)),
    ("flow_band", "string", dict(values=["<10k", "10k-100k", "100k-1m", ">1m"], weights=[30, 40, 22, 8], random=True)),
    U("u")])
smr = (smr.join(slc.select("idx", "acn", "distressed"), smr.company_idx == slc.idx)
          .filter((F.col("distressed") == 1) | (F.col("u") < 0.35))   # distressed entities attract more SMRs
          .select(F.format_string("SMR%08d", "id").alias("smr_id"), "acn", "report_date", "smr_type", "flow_band",
                  F.lit("AUSTRAC-RESTRICTED").alias("classification")))
write(smr, "austrac_smr")

flags = F.array(*[F.lit(x) for x in ["Capital adequacy concern", "Governance review", "Failed performance test", "Liquidity watch"]])
apra = (slc.filter("is_licensee = 1").select("acn", "collapsed")
        .withColumn("u", F.rand(SEED))
        .filter(F.col("u") < 0.1 + 0.3 * F.col("collapsed"))
        .withColumn("prudential_flag", F.element_at(flags, (F.floor(F.rand(SEED + 1) * 4) + 1).cast("int")))
        .withColumn("flag_date", F.date_sub(F.current_date(), (F.rand(SEED + 2) * 900).cast("int")))
        .select("acn", "prudential_flag", "flag_date"))
write(apra, "apra_prudential_flags")

In [0]:
def pick(values, seed):
    arr = F.array(*[F.lit(v) for v in values])
    return F.element_at(arr, (F.floor(F.rand(seed) * len(values)) + 1).cast("int"))

# P15 AFCA complaints by licensee and year
years = spark.range(2021, 2027).withColumnRenamed("id", "year")
afca = (slc.filter("is_licensee = 1").select("acn", "collapsed").crossJoin(years)
        .withColumn("complaints", F.floor(F.exp(F.rand(SEED) * 4 + F.col("collapsed") * 1.2)).cast("int"))
        .withColumn("complaints_upheld_pct", F.round(0.2 + F.rand(SEED + 1) * 0.5, 3))
        .withColumn("top_issue", pick(["Fees and charges", "Incorrect advice", "Delay in claim", "Responsible lending", "Service quality"], SEED + 2))
        .drop("collapsed"))
write(afca, "afca_complaints")

# P8 enforceable undertakings, infringement notices, MDP outcomes
alt = gen("alternatives", max(int(N * 0.06), 50), [
    ("company_idx", "long", dict(minValue=0, maxValue=N - 1, random=True)),
    ("outcome_type", "string", dict(values=["Court enforceable undertaking", "Infringement notice", "Markets Disciplinary Panel"], weights=[35, 55, 10], random=True)),
    ("provision", "string", dict(values=["s912A Corporations Act", "s1041H Corporations Act", "s12DA ASIC Act", "s47 National Credit Act", "s180 Corporations Act"], random=True)),
    ("outcome_date", "date", dict(data_range=dg.DateRange("2015-01-01 00:00:00", "2026-09-30 00:00:00", "days=1"), random=True)),
    ("amount_aud", "double", dict(minValue=13320, maxValue=5000000, step=10, random=True))])
alt = (alt.join(slc.select("idx", "acn"), alt.company_idx == slc.idx)
          .select(F.format_string("ALT%06d", "id").alias("outcome_id"), "acn", "outcome_type", "provision", "outcome_date",
                  F.round("amount_aud", 0).alias("amount_aud")))
write(alt, "enforcement_alternatives")

# P7 published insolvency / deregistration notices, dated before the collapse
notices = (slc.filter("collapsed = 1").select("acn", "collapse_date")
           .withColumn("n", F.explode(F.sequence(F.lit(1), (F.floor(F.rand(SEED) * 3) + 1).cast("int"))))
           .withColumn("notice_type", pick(["Appointment of administrator", "Appointment of liquidator", "Meeting of creditors", "Winding up application", "Proposed deregistration"], SEED + 3))
           .withColumn("notice_date", F.date_sub("collapse_date", (F.col("n") * 30 + F.rand(SEED + 4) * 60).cast("int")))
           .select(F.substring(F.sha2(F.concat_ws("|", "acn", "n"), 256), 1, 16).alias("notice_id"), "acn", "notice_type", "notice_date"))
write(notices, "published_notices")

In [0]:
br = gen("breach_reports", max(int(N * 0.16), 100), [
    ("company_idx", "long", dict(minValue=0, maxValue=N - 1, random=True)),
    ("report_kind", "string", dict(values=["Breach report", "Report of misconduct", "Liquidator report", "Market surveillance alert"], weights=[40, 35, 15, 10], random=True)),
    ("category", "string", dict(values=["Misleading conduct", "Unlicensed credit", "Director duties", "Phoenix activity", "Super mis-selling", "Insider trading", "Failure to lodge"],
                                 weights=[20, 12, 18, 14, 10, 6, 20], random=True)),
    ("received_at", "timestamp", dict(data_range=dg.DateRange("2025-05-01 00:00:00", "2026-11-05 23:59:00", "minutes=7"), random=True)),
    ("consumers_affected", "int", dict(minValue=0, maxValue=5000, random=True)),
    ("alleged_loss_aud", "double", dict(minValue=0, maxValue=20000000, step=100, random=True)),
    ("triage_status", "string", dict(values=["New", "In triage", "Case opened", "Linked to case", "No further action"], weights=[35, 20, 15, 10, 20], random=True))])
br = (br.join(slc.select("idx", "acn", "company_name"), br.company_idx == slc.idx)
        .withColumn("summary", F.format_string("%s received regarding %s (ACN %s): alleged %s affecting approx. %d consumers.",
                                               "report_kind", "company_name", "acn", F.lower("category"), "consumers_affected"))
        .select(F.format_string("INT-%06d", "id").alias("intake_id"), "acn", "report_kind", "category", "received_at",
                "consumers_affected", F.round("alleged_loss_aud", 0).alias("alleged_loss_aud"), "triage_status", "summary"))
write(br, "breach_reports")

In [0]:
# Calibration data for P(success), cost and recovery. Real public outcomes (P10 judgments, P4 bans, P8) are joined
# downstream; internal costs and recoveries are never public, so they are synthetic and flagged as estimates.
hm = gen("historical_matters", 1500, [
    ("company_idx", "long", dict(minValue=0, maxValue=N - 1, random=True)),
    ("matter_type", "string", dict(values=["Civil penalty", "Criminal referral", "Administrative", "Infringement"], weights=[45, 15, 30, 10], random=True)),
    ("filed_date", "date", dict(data_range=dg.DateRange("2012-01-01 00:00:00", "2024-06-30 00:00:00", "days=1"), random=True)),
    U("u_dur"), U("u_cost"), U("u_ev"), U("u_succ"), U("u_rec"), U("u_amt")])
type_cost = (F.when(F.col("matter_type") == "Criminal referral", 1.6).when(F.col("matter_type") == "Civil penalty", 1.0)
              .when(F.col("matter_type") == "Administrative", 0.35).otherwise(0.1))
hm = (hm.join(slc.select("idx", "acn", "collapse_date"), hm.company_idx == slc.idx)
    .withColumn("evidence_strength", F.round("u_ev", 3))
    .withColumn("duration_months", F.round(4 + F.col("u_dur") * 44 * type_cost).cast("int"))
    .withColumn("judgment_date", F.expr("add_months(filed_date, duration_months)"))
    .withColumn("litigation_cost_aud", F.round(type_cost * 120000 * F.exp(F.col("u_cost") * 1.5) * (F.col("duration_months") / 12.0), 0))
    .withColumn("claimed_amount_aud", F.round(F.exp(11 + F.col("u_amt") * 5), 0))
    .withColumn("success", (F.col("u_succ") < 0.25 + 0.65 * F.col("evidence_strength")).cast("int"))
    .withColumn("collapsed_before_judgment", (F.col("collapse_date").isNotNull() & (F.col("collapse_date") <= F.col("judgment_date"))).cast("int"))
    .withColumn("recovered_amount_aud", F.round(F.col("claimed_amount_aud") * F.col("success") * (1 - F.col("collapsed_before_judgment") * 0.9) * (0.2 + 0.7 * F.col("u_rec")), 0))
    .withColumn("penalty_aud", F.round(F.col("success") * F.col("claimed_amount_aud") * 0.3, 0))
    .select(F.format_string("HM-%05d", "id").alias("matter_id"), "acn", "matter_type", "filed_date", "judgment_date", "duration_months",
            "evidence_strength", "litigation_cost_aud", "claimed_amount_aud", "success", "collapsed_before_judgment",
            "recovered_amount_aud", "penalty_aud", F.lit(True).alias("recovery_is_estimate")))
write(hm, "historical_matters")

In [0]:
teams = spark.createDataFrame([
    ("T01", "Financial Services Enforcement"), ("T02", "Corporations and Insolvency"),
    ("T03", "Superannuation and Investment"), ("T04", "Credit and Consumer")], "team_id string, team_name string")
users = spark.createDataFrame([
    ("u.investigator1", "Priya Nair", "Investigator", "T02"), ("u.investigator2", "Sam Okafor", "Investigator", "T01"),
    ("u.investigator3", "Ana Costa", "Investigator", "T03"), ("u.investigator4", "Tom Becker", "Investigator", "T04"),
    ("u.lead1", "Marcus Lee", "Enforcement Lead", "T02"), ("u.lead2", "Jo Whitfield", "Enforcement Lead", "T01"),
    ("u.lead3", "Kim Tran", "Enforcement Lead", "T03"), ("u.lead4", "Ravi Menon", "Enforcement Lead", "T04"),
    ("u.counsel1", "Helen Ward", "Legal Counsel", "T02"), ("u.counsel2", "Ben Adler", "Legal Counsel", "T01")],
    "user_id string, display_name string, persona string, team_id string")
write(teams, "teams"); write(users, "users")

In [0]:
# J2 phoenix story: a director who controls >= 2 collapsed companies and at least one live one
d_ = spark.read.parquet(f"{RAW_ROOT}/synthetic/directorships").join(slc.select("acn", "collapsed", "status"), "acn")
phoenix_person = (d_.groupBy("person_id")
                  .agg(F.sum("collapsed").alias("n_col"), F.sum(F.expr("int(status = 'REGD')")).alias("n_live"))
                  .filter("n_col >= 2 AND n_live >= 1").orderBy(F.desc("n_col"), "person_id").first())
assert phoenix_person, "no phoenix story candidate - increase curated_slice_size"
phoenix_acn = (d_.filter((F.col("person_id") == phoenix_person.person_id) & (F.col("status") == "REGD"))
                 .orderBy("acn").first().acn)
live = slc.filter("status = 'REGD'")
super_acn = (live.filter("is_licensee = 1").withColumn("m", F.col("company_name").rlike("SUPER").cast("int"))
                 .orderBy(F.desc("m"), "acn").first().acn)
credit_acn = (live.filter("is_licensee = 0").withColumn("m", F.col("company_name").rlike("FINANCE|CREDIT|LOAN|LENDING|CASH").cast("int"))
                  .orderBy(F.desc("m"), "acn").first().acn)
story = spark.createDataFrame([
    (0, phoenix_acn, "phoenix_network", "Civil penalty", "T02", "Phoenix activity and director duties"),
    (1, super_acn, "super_mis_selling", "Civil penalty", "T03", "Superannuation underperformance and mis-selling"),
    (2, credit_acn, "unlicensed_credit", "Criminal referral", "T04", "Unlicensed credit activity")],
    "case_idx long, story_acn string, story_tag string, s_type string, s_team string, s_title string")
print("phoenix director", phoenix_person.person_id, "collapsed cos", phoenix_person.n_col)
print("story respondents:", phoenix_acn, super_acn, credit_acn)

In [0]:
cases = gen("cases", NUM_CASES, [
    ("case_type", "string", dict(values=["Civil penalty", "Criminal referral", "Administrative", "Infringement"], weights=[45, 15, 30, 10], random=True)),
    ("stage", "string", dict(values=["Assessment", "Investigation", "Decision", "Litigation", "Referral", "Outcome"], weights=[20, 35, 20, 15, 5, 5], random=True)),
    ("priority", "string", dict(values=["P1", "P2", "P3"], weights=[20, 50, 30], random=True)),
    ("team_id", "string", dict(values=["T01", "T02", "T03", "T04"], random=True)),
    ("opened_at", "timestamp", dict(data_range=dg.DateRange("2024-07-01 00:00:00", "2026-10-31 00:00:00", "hours=5"), random=True)),
    ("pick", "long", dict(minValue=0, maxValue=1000000, random=True)),
    U("u_rec"), U("u_cost"), U("u_dur"), U("u_harm"), U("u_det"), U("u_ev")]).withColumnRenamed("id", "case_idx")

live_ix = live.select(F.col("acn").alias("rand_acn"), F.row_number().over(W.orderBy(F.xxhash64("acn", F.lit(SEED + 9)))).alias("rn")).cache()
n_live = live_ix.count()
leads = users.filter("persona = 'Enforcement Lead'").select("team_id", F.col("user_id").alias("lead_user"))

cases = (cases.withColumn("rn", (F.col("pick") % n_live) + 1).join(live_ix, "rn", "left")
    .join(story, "case_idx", "left")
    .withColumn("respondent_acn", F.coalesce("story_acn", "rand_acn"))
    .withColumn("case_type", F.coalesce("s_type", "case_type"))
    .withColumn("team_id", F.coalesce("s_team", "team_id"))
    .withColumn("est_duration_months", F.round(8 + F.col("u_dur") * 30).cast("int"))
    .withColumn("est_recoverable_aud", F.round(F.exp(13 + F.col("u_rec") * 4), -3))
    .withColumn("est_litigation_cost_aud", F.round(150000 * F.exp(F.col("u_cost") * 1.5) * F.col("est_duration_months") / 12.0, -3))
    .withColumn("cost_to_date_aud", F.round(F.col("est_litigation_cost_aud") * (0.05 + F.col("u_cost") * 0.4), -2))
    .withColumn("consumers_affected", F.round(F.exp(F.col("u_harm") * 9)).cast("int"))
    .withColumn("total_consumer_loss_aud", F.round(F.col("consumers_affected") * (500 + F.col("u_rec") * 20000), -2))
    .withColumn("vulnerable_cohort", F.col("u_harm") > 0.6)
    .withColumn("strategic_priority_score", F.round(F.col("u_det") * 100).cast("int"))
    .withColumn("counsel_evidence_strength", F.when(F.col("stage").isin("Decision", "Litigation", "Referral", "Outcome"),
                                                    F.round(0.3 + F.col("u_ev") * 0.6, 2)))
    .withColumn("limitation_date", F.add_months(F.to_date("opened_at"), 72))
    .withColumn("case_id", F.format_string("C%05d", "case_idx"))
    .withColumn("case_number", F.format_string("ASIC-%d-%04d", F.year("opened_at"), (F.col("case_idx") + 1).cast("int")))
    .join(slc.select(F.col("acn").alias("respondent_acn"), "company_name"), "respondent_acn")
    .withColumn("title", F.concat_ws(" - ", "company_name", F.coalesce("s_title", "case_type")))
    .join(leads, "team_id", "left")
    .select("case_id", "case_number", "title", "case_type", "stage", "priority", "team_id", "lead_user", "opened_at",
            "respondent_acn", "story_tag", "est_recoverable_aud", "est_litigation_cost_aud", "est_duration_months",
            "cost_to_date_aud", "consumers_affected", "total_consumer_loss_aud", "vulnerable_cohort",
            "strategic_priority_score", "counsel_evidence_strength", "limitation_date"))
write(cases, "cases")

# Parties: respondent company, its directors (respondent persons) and witnesses
c = spark.read.parquet(f"{RAW_ROOT}/synthetic/cases").select("case_id", "respondent_acn")
dirs = spark.read.parquet(f"{RAW_ROOT}/synthetic/directorships")
ppl = spark.read.parquet(f"{RAW_ROOT}/synthetic/persons").select("person_id", "person_idx")
witness = (c.withColumn("k", F.explode(F.sequence(F.lit(1), F.lit(3))))
            .withColumn("person_idx", (F.abs(F.xxhash64("case_id", "k", F.lit(SEED))) % NP).cast("long"))
            .join(ppl, "person_idx")
            .select("case_id", F.lit("person").alias("party_kind"), F.col("person_id").alias("party_ref"), F.lit("witness").alias("role")))
parties = (c.select("case_id", F.lit("company").alias("party_kind"), F.col("respondent_acn").alias("party_ref"), F.lit("respondent").alias("role"))
    .unionByName(c.join(dirs, c.respondent_acn == dirs.acn)
                  .select("case_id", F.lit("person").alias("party_kind"), F.col("person_id").alias("party_ref"), F.lit("respondent").alias("role")))
    .unionByName(witness).dropDuplicates(["case_id", "party_kind", "party_ref"]))
write(parties, "case_parties")

In [0]:
chk = spark.read.parquet(f"{RAW_ROOT}/synthetic/cases")
assert chk.count() == NUM_CASES, "case count mismatch"
assert chk.filter("story_tag IS NOT NULL").count() == 3, "story cases missing"
assert chk.filter("respondent_acn IS NULL OR lead_user IS NULL").count() == 0
# Signal check: synthetic ATO debt should be higher for companies that really collapsed
display(spark.read.parquet(f"{RAW_ROOT}/synthetic/ato_tax_profile").join(slc.select("acn", "collapsed"), "acn")
        .groupBy("collapsed").agg(F.avg("ato_debt_aud").alias("avg_debt"), F.avg("lodgement_on_time_pct").alias("avg_lodge")))